# W06 Extension — Feature Engineering and Ranking Objective

This notebook tests whether defensible feature engineering or an advanced ranking objective can improve the current LightGBM result. Every candidate uses the same five `GroupKFold(client_id)` folds, and the rule baseline is recomputed on those exact test rows.

> Scope: cross-client ranking in the 30k starter snapshot. This is **not** evidence of future decline forecasting because inherited 90-day features overlap the current decline-label window.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

import numpy as np
import pandas as pd

def find_repo_root(start):
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'skills' / 'README.md').exists():
            return candidate
    raise FileNotFoundError('Could not locate repository root')

ROOT = find_repo_root(Path.cwd())
SCRIPT = ROOT / 'work' / 'scripts' / 'feature_engineering_experiment.py'
RESULTS = ROOT / 'work' / 'outputs' / 'feature_engineering_results.json'
OOF = ROOT / 'work' / 'outputs' / 'feature_engineering_oof.csv'
FEATURES = ROOT / 'data' / 'processed' / 'refresh_feature_vector.csv'

print(f'Repository: {ROOT}')
print(f'Experiment script present: {SCRIPT.exists()}')
print(f'Python: {sys.version.split()[0]}')

Repository: C:\Local D\Galeri Belajar\Project Code\FlyRank Internship
Experiment script present: True
Python: 3.13.9


## Experimental design

- **R0:** rule baseline, evaluated inside each grouped fold.
- **E0:** exact current W06 LightGBM and feature set.
- **E1:** missingness, intensity, interaction, and within-client relative features with the same LightGBM.
- **E2:** engineered features with a slower, regularized LightGBM.
- **E3:** current features with LambdaRank, optimizing within-client editorial queues.
- **E4:** engineered features with LambdaRank.
- **E5:** fixed rank blend of E1 and E2.

Primary metric: mean P@50 across the five fixed folds. P@20, P@100, AP, ROC-AUC, lift, and fold dispersion remain visible so a narrow top-K gain is not presented as universal improvement.

A previous-window feature family was tested and rejected before this final run: combined with overlapping 90-day totals, it produced implausibly near-perfect scores by partially reconstructing the label. Those columns are not present below.

In [2]:
completed = subprocess.run([sys.executable, '-X', 'utf8', str(SCRIPT)], cwd=ROOT, check=True, capture_output=True, text=True)
print(completed.stdout)
print(f'Experiment exit code: {completed.returncode}')

Rows: 30,000; clients: 32; positive rate: 54.21%
Base matrix: (30000, 52); engineered matrix: (30000, 80)
Added engineered numeric features: 28
Forbidden-column guard: PASS

=== R0_rule_baseline ===
fold=1 rows=7,008 clients=1 base=49.0% P@20=35.0% P@50=38.0% P@100=34.0% AUC=0.5522 AP=0.5134
fold=2 rows=5,731 clients=7 base=64.5% P@20=40.0% P@50=42.0% P@100=39.0% AUC=0.4436 AP=0.5949
fold=3 rows=5,753 clients=8 base=37.9% P@20=35.0% P@50=28.0% P@100=34.0% AUC=0.6115 AP=0.4203
fold=4 rows=5,755 clients=8 base=62.2% P@20=55.0% P@50=56.0% P@100=56.0% AUC=0.7142 AP=0.7350
fold=5 rows=5,753 clients=8 base=58.5% P@20=70.0% P@50=68.0% P@100=69.0% AUC=0.6800 AP=0.7161
MEAN P@50=46.4% +/- 15.7%; AUC=0.6003; AP=0.5959

=== E0_current_lgbm ===
fold=1 rows=7,008 clients=1 base=49.0% P@20=85.0% P@50=88.0% P@100=87.0% AUC=0.6480 AP=0.6391
fold=2 rows=5,731 clients=7 base=64.5% P@20=85.0% P@50=80.0% P@100=73.0% AUC=0.6062 AP=0.7139
fold=3 rows=5,753 clients=8 base=37.9% P@20=85.0% P@50=82.0% P@100=79

In [3]:
payload = json.loads(RESULTS.read_text(encoding='utf-8'))
comparison = pd.DataFrame(payload['comparison']).sort_values('mean_p_at_50', ascending=False)
shown = comparison[[
    'candidate', 'mean_p_at_20', 'mean_p_at_50', 'std_p_at_50',
    'mean_p_at_100', 'mean_lift_at_50', 'mean_roc_auc',
    'mean_average_precision'
]].copy()
for column in ['mean_p_at_20', 'mean_p_at_50', 'std_p_at_50', 'mean_p_at_100']:
    shown[column] = (shown[column] * 100).round(1).astype(str) + '%'
shown['mean_lift_at_50'] = shown['mean_lift_at_50'].round(2)
shown['mean_roc_auc'] = shown['mean_roc_auc'].round(4)
shown['mean_average_precision'] = shown['mean_average_precision'].round(4)
print(shown.reset_index(drop=True).to_string(index=False))

assert payload['winner_mean_p_at_50'] > payload['current_mean_p_at_50']
print(f"Winner: {payload['winner']}")
print(f"Current mean P@50: {payload['current_mean_p_at_50']:.1%}")
print(f"Winner mean P@50:  {payload['winner_mean_p_at_50']:.1%}")
print(f"Paired improvement: {payload['paired_improvement_pp']:+.1f}pp")

                     candidate mean_p_at_20 mean_p_at_50 std_p_at_50 mean_p_at_100  mean_lift_at_50  mean_roc_auc  mean_average_precision
      E4_engineered_lambdarank        94.0%        91.6%        4.6%         85.0%             1.74        0.6280                  0.6703
            E3_base_lambdarank        90.0%        90.8%        2.7%         90.4%             1.73        0.6467                  0.6846
E2_engineered_regularized_lgbm        90.0%        86.8%        4.8%         85.0%             1.65        0.6753                  0.6904
      E5_engineered_rank_blend        86.0%        86.4%        6.4%         82.8%             1.63        0.6701                  0.6839
               E0_current_lgbm        86.0%        84.0%        3.7%         78.8%             1.60        0.6787                  0.6869
       E1_engineered_same_lgbm        86.0%        83.2%        6.4%         80.2%             1.58        0.6594                  0.6701
              R0_rule_baseline    

In [4]:
current_folds = pd.DataFrame(payload['results']['E0_current_lgbm']['folds'])
winner_folds = pd.DataFrame(payload['results'][payload['winner']]['folds'])
paired = current_folds[['fold', 'test_rows', 'test_clients', 'base_rate']].copy()
paired['current_p50'] = current_folds['p_at_50']
paired['winner_p50'] = winner_folds['p_at_50']
paired['improvement_pp'] = 100 * (paired['winner_p50'] - paired['current_p50'])
paired['current_p20'] = current_folds['p_at_20']
paired['winner_p20'] = winner_folds['p_at_20']
paired_shown = paired.copy()
for column in ['base_rate', 'current_p50', 'winner_p50', 'current_p20', 'winner_p20']:
    paired_shown[column] = (paired_shown[column] * 100).round(1).astype(str) + '%'
paired_shown['improvement_pp'] = paired_shown['improvement_pp'].map(lambda value: f'{value:+.1f}')
print(paired_shown.to_string(index=False))
print(f"Winner improved P@50 in {(paired['improvement_pp'] > 0).sum()} of {len(paired)} folds.")

 fold  test_rows  test_clients base_rate current_p50 winner_p50 improvement_pp current_p20 winner_p20
    1       7008             1     49.0%       88.0%      86.0%           -2.0       85.0%      85.0%
    2       5731             7     64.5%       80.0%      90.0%          +10.0       85.0%     100.0%
    3       5753             8     37.9%       82.0%      90.0%           +8.0       85.0%      90.0%
    4       5755             8     62.2%       82.0%      94.0%          +12.0       85.0%      95.0%
    5       5753             8     58.5%       88.0%      98.0%          +10.0       90.0%     100.0%
Winner improved P@50 in 4 of 5 folds.


In [5]:
importance = pd.DataFrame(payload['top_feature_importance'][payload['winner']]).head(15)
print(importance.to_string(index=False, formatters={'mean_importance': '{:.1f}'.format}))
print('Top three:', ', '.join(importance['feature'].head(3)))
print('Sanity check: no label, ID, latest-30-day outcome, or product-score column appears.')

                       feature mean_importance
         days_with_impressions          1027.2
              content_age_days           919.0
                  avg_position           775.0
    client_position_percentile           731.2
log_impressions_per_active_day           725.8
 client_impressions_percentile           718.4
         client_position_delta           708.4
         position_x_visibility           661.2
        visibility_x_staleness           635.6
      client_impressions_delta           633.8
    client_sessions_percentile           515.2
   client_staleness_percentile           490.2
                    word_count           464.8
  client_engagement_percentile           414.8
                    char_count           409.4
Top three: days_with_impressions, content_age_days, avg_position
Sanity check: no label, ID, latest-30-day outcome, or product-score column appears.


In [6]:
oof = pd.read_csv(OOF)
features = pd.read_csv(FEATURES)
winner = payload['winner']

queue_rows = []
for fold, fold_frame in oof.groupby('fold'):
    current_top = fold_frame.nlargest(50, 'E0_current_lgbm')
    winner_top = fold_frame.nlargest(50, winner)
    queue_rows.append({
        'fold': int(fold),
        'current_true_positives': int(current_top['is_declining_label'].sum()),
        'winner_true_positives': int(winner_top['is_declining_label'].sum()),
        'additional_true_positives': int(winner_top['is_declining_label'].sum() - current_top['is_declining_label'].sum()),
    })
queue_audit = pd.DataFrame(queue_rows)
print(queue_audit.to_string(index=False))
print(f"Across the five 50-page queues, the winner retrieves {queue_audit['additional_true_positives'].sum():+d} additional positives.")

winner_top_indices = []
for _, fold_frame in oof.groupby('fold'):
    winner_top_indices.extend(fold_frame.nlargest(50, winner).index.tolist())
false_positive_indices = [i for i in winner_top_indices if int(oof.loc[i, 'is_declining_label']) == 0]
false_positive_indices = sorted(false_positive_indices, key=lambda i: oof.loc[i, winner], reverse=True)[:3]
case_columns = ['content_type', 'impression_tier', 'position_tier', 'freshness_tier', 'avg_position', 'days_since_last_update']
cases = features.loc[false_positive_indices, case_columns].copy().reset_index(drop=True)
cases.insert(0, 'case', [f'FP case {i+1}' for i in range(len(cases))])
cases['why_hard'] = 'High queue score but the threshold label is not down; inspect volatility, seasonality, and current editorial context.'
print(cases.to_string(index=False))
print('Cases are anonymous: no client name, URL, query, or identifier is displayed.')

 fold  current_true_positives  winner_true_positives  additional_true_positives
    1                      44                     43                         -1
    2                      40                     45                          5
    3                      41                     45                          4
    4                      41                     47                          6
    5                      44                     49                          5
Across the five 50-page queues, the winner retrieves +19 additional positives.
     case    content_type impression_tier position_tier freshness_tier  avg_position  days_since_last_update                                                                                                              why_hard
FP case 1 keyword article        moderate        page_1         91-180           4.7                     104 High queue score but the threshold label is not down; inspect volatility, seasonality, and current editor

## Honest conclusion

On the five fixed client-grouped development folds, engineered LambdaRank produced the highest observed mean P@50: **91.6% ± 4.6%**, compared with **84.0% ± 3.7%** for the current LightGBM classifier. The paired gain is **+7.6 percentage points**, with improvement in four of five folds.

Most of the gain comes from the ranking objective: base-feature LambdaRank reached 90.8% P@50. The engineered features added a smaller 0.8pp at K=50. The winner also improved P@20, but its ROC-AUC and Average Precision were lower than the current classifier, so it should be described as a better **top-of-queue specialist**, not a universally better probability model.

This is a development-CV result selected on these folds, not a locked final test. It supports the observed cross-client current-snapshot ranking claim only. Future decline forecasting still requires pre-T features, post-T labels, and temporal validation on the daily warehouse.

## Self-check

- [x] Rule baseline and every model use the same folds and metrics.
- [x] Forbidden label, ID, outcome-window, and product-score columns are guarded.
- [x] An implausibly strong feature family was rejected rather than reported.
- [x] Fold dispersion, secondary metrics, feature importance, and errors are visible.
- [x] Notebook runs top to bottom with fixed seeds.